# 36 - JS (Transformers.js) 埋め込みを DuckDB に保存

## 概要
Notebook 31 で生成した JavaScript (Transformers.js / ONNX Runtime) の SigLIP 画像埋め込みを
DuckDB に保存する。既存の `image_embeddings` テーブルに、model_name を区別して格納する。

## 保存する埋め込み

| dtype | model_name (DuckDB) | 画像数 | ソース |
|-------|---------------------|--------|--------|
| fp32 | `Xenova/siglip-base-patch16-224` | 378 | Node.js ONNX Runtime |
| fp16 | `Xenova/siglip-base-patch16-224-fp16` | 378 | Node.js ONNX Runtime |
| q8 | `Xenova/siglip-base-patch16-224-q8` | 378 | Node.js ONNX Runtime |
| q4 | `Xenova/siglip-base-patch16-224-q4` | 378 | Node.js ONNX Runtime |
| fp32 (browser) | `Xenova/siglip-base-patch16-224-browser` | 50 | Chromium WASM |

## テーブル構造
既存の `image_embeddings` テーブルを使用（`(id, model_name)` が複合PK）

In [1]:
import json
from pathlib import Path

import duckdb
import numpy as np
import pandas as pd

DB_PATH = Path("../data/images.duckdb")
JS_DIR = Path("../data/js_embeddings")

# JS embedding files and their model names in DuckDB
JS_MODELS = {
    "vision_fp32.json": "Xenova/siglip-base-patch16-224",
    "vision_fp16.json": "Xenova/siglip-base-patch16-224-fp16",
    "vision_q8.json": "Xenova/siglip-base-patch16-224-q8",
    "vision_q4.json": "Xenova/siglip-base-patch16-224-q4",
    "browser_vision_fp32.json": "Xenova/siglip-base-patch16-224-browser",
}

print(f"DB: {DB_PATH}")
print(f"JS dir: {JS_DIR}")
print(f"Models to insert: {len(JS_MODELS)}")

DB: ../data/images.duckdb
JS dir: ../data/js_embeddings
Models to insert: 5


## 1. 現在のテーブル状態を確認

In [2]:
conn = duckdb.connect(str(DB_PATH))

print("現在の image_embeddings テーブル:")
print(conn.execute("""
    SELECT model_name, COUNT(*) as count
    FROM image_embeddings
    GROUP BY model_name
    ORDER BY model_name
""").fetchdf().to_string(index=False))

print(f"\nTotal rows: {conn.execute('SELECT COUNT(*) FROM image_embeddings').fetchone()[0]}")
conn.close()

現在の image_embeddings テーブル:
                    model_name  count
google/siglip-base-patch16-224    378
 openai/clip-vit-large-patch14    378

Total rows: 756


## 2. JS 埋め込みデータの読み込みと検証

In [3]:
# 各JSONファイルを読み込んで検証
js_datasets = {}

for filename, model_name in JS_MODELS.items():
    path = JS_DIR / filename
    if not path.exists():
        print(f"  SKIP: {filename} not found")
        continue
    
    with open(path) as f:
        data = json.load(f)
    
    embeddings = data["embeddings"]
    n = len(embeddings)
    dim = len(embeddings[0]["embedding"]) if n > 0 else 0
    
    # エラーのある埋め込みをフィルタ
    valid = [e for e in embeddings if not e.get("error")]
    errors = [e for e in embeddings if e.get("error")]
    
    # L2ノルム検証
    norms = [np.linalg.norm(e["embedding"]) for e in valid]
    
    print(f"  {filename}:")
    print(f"    model_name: {model_name}")
    print(f"    total: {n}, valid: {len(valid)}, errors: {len(errors)}")
    print(f"    dim: {dim}")
    print(f"    L2 norm: mean={np.mean(norms):.6f}, min={np.min(norms):.6f}, max={np.max(norms):.6f}")
    
    js_datasets[model_name] = valid

print(f"\nDatasets ready: {len(js_datasets)}")

  vision_fp32.json:
    model_name: Xenova/siglip-base-patch16-224
    total: 378, valid: 378, errors: 0
    dim: 768
    L2 norm: mean=1.000000, min=1.000000, max=1.000000
  vision_fp16.json:
    model_name: Xenova/siglip-base-patch16-224-fp16
    total: 378, valid: 378, errors: 0
    dim: 768
    L2 norm: mean=1.000000, min=1.000000, max=1.000000


  vision_q8.json:
    model_name: Xenova/siglip-base-patch16-224-q8
    total: 378, valid: 378, errors: 0
    dim: 768
    L2 norm: mean=1.000000, min=1.000000, max=1.000000


  vision_q4.json:
    model_name: Xenova/siglip-base-patch16-224-q4
    total: 378, valid: 378, errors: 0
    dim: 768
    L2 norm: mean=1.000000, min=1.000000, max=1.000000
  browser_vision_fp32.json:
    model_name: Xenova/siglip-base-patch16-224-browser
    total: 50, valid: 50, errors: 0
    dim: 768
    L2 norm: mean=1.000000, min=1.000000, max=1.000000

Datasets ready: 5


## 3. DuckDB にデータを挿入

In [4]:
conn = duckdb.connect(str(DB_PATH))

# HNSW index requires vss extension
conn.execute("INSTALL vss")
conn.execute("LOAD vss")

total_inserted = 0

for model_name, embeddings in js_datasets.items():
    # 既存データがあれば削除（再実行対応）
    existing = conn.execute(
        "SELECT COUNT(*) FROM image_embeddings WHERE model_name = ?",
        [model_name]
    ).fetchone()[0]
    
    if existing > 0:
        print(f"  {model_name}: deleting {existing} existing rows...")
        conn.execute("DELETE FROM image_embeddings WHERE model_name = ?", [model_name])
    
    # バッチ挿入
    rows = []
    for e in embeddings:
        rows.append((e["id"], model_name, e["embedding"]))
    
    conn.executemany(
        "INSERT INTO image_embeddings (id, model_name, embedding) VALUES (?, ?, ?)",
        rows
    )
    
    total_inserted += len(rows)
    print(f"  {model_name}: inserted {len(rows)} rows")

conn.close()
print(f"\nTotal inserted: {total_inserted} rows")

  Xenova/siglip-base-patch16-224: inserted 378 rows


  Xenova/siglip-base-patch16-224-fp16: inserted 378 rows


  Xenova/siglip-base-patch16-224-q8: inserted 378 rows


  Xenova/siglip-base-patch16-224-q4: inserted 378 rows
  Xenova/siglip-base-patch16-224-browser: inserted 50 rows

Total inserted: 1562 rows


## 4. 挿入結果の確認

In [5]:
conn = duckdb.connect(str(DB_PATH), read_only=True)
conn.execute("LOAD vss")

print("image_embeddings テーブル（挿入後）:")
result_df = conn.execute("""
    SELECT model_name, COUNT(*) as count,
           MIN(created_at) as first_created,
           MAX(created_at) as last_created
    FROM image_embeddings
    GROUP BY model_name
    ORDER BY model_name
""").fetchdf()
display(result_df)

print(f"\nTotal rows: {conn.execute('SELECT COUNT(*) FROM image_embeddings').fetchone()[0]}")

image_embeddings テーブル（挿入後）:


,model_name,count,first_created,last_created
0,Xenova/siglip-base-patch16-224,378,2026-02-13 00:44:20.170414,2026-02-13 00:44:20.947568
1,Xenova/siglip-base-patch16-224-browser,50,2026-02-13 00:44:23.521767,2026-02-13 00:44:23.642167
2,Xenova/siglip-base-patch16-224-fp16,378,2026-02-13 00:44:20.950494,2026-02-13 00:44:21.750141
3,Xenova/siglip-base-patch16-224-q4,378,2026-02-13 00:44:22.608165,2026-02-13 00:44:23.518460
4,Xenova/siglip-base-patch16-224-q8,378,2026-02-13 00:44:21.752979,2026-02-13 00:44:22.605103
5,google/siglip-base-patch16-224,378,2026-02-03 11:56:02.162628,2026-02-03 11:57:39.480309
6,openai/clip-vit-large-patch14,378,2026-02-03 12:46:36.033171,2026-02-03 12:48:18.349432



Total rows: 2318


## 5. 埋め込みのサニティチェック（DuckDB読み込み検証）

In [6]:
# DuckDBに保存したデータとJSONファイルの一致を確認
print("サニティチェック: DuckDB vs JSON ファイル")
print("=" * 60)

for model_name in js_datasets:
    # DuckDBから読み込み
    db_df = conn.execute("""
        SELECT id, embedding
        FROM image_embeddings
        WHERE model_name = ?
        ORDER BY id
    """, [model_name]).fetchdf()
    
    db_embs = {row.id: np.array(row.embedding, dtype=np.float32) for row in db_df.itertuples()}
    
    # JSONと比較
    json_embs = {e["id"]: np.array(e["embedding"], dtype=np.float32) for e in js_datasets[model_name]}
    
    # 共通IDで比較
    common_ids = set(db_embs.keys()) & set(json_embs.keys())
    diffs = []
    for id_ in common_ids:
        diff = np.abs(db_embs[id_] - json_embs[id_]).max()
        diffs.append(diff)
    
    max_diff = max(diffs)
    match = max_diff < 1e-6
    
    status = "OK" if match else "MISMATCH"
    print(f"  {model_name}: {len(common_ids)} embeddings, max_diff={max_diff:.2e} [{status}]")

conn.close()
print("\nDone.")

サニティチェック: DuckDB vs JSON ファイル
  Xenova/siglip-base-patch16-224: 378 embeddings, max_diff=0.00e+00 [OK]
  Xenova/siglip-base-patch16-224-fp16: 378 embeddings, max_diff=0.00e+00 [OK]
  Xenova/siglip-base-patch16-224-q8: 378 embeddings, max_diff=0.00e+00 [OK]
  Xenova/siglip-base-patch16-224-q4: 378 embeddings, max_diff=0.00e+00 [OK]
  Xenova/siglip-base-patch16-224-browser: 50 embeddings, max_diff=0.00e+00 [OK]

Done.


## 6. クロス環境での検索テスト

In [7]:
# Python SigLIP vs JS SigLIP (各dtype) でのクロス検索テスト
conn = duckdb.connect(str(DB_PATH), read_only=True)
conn.execute("LOAD vss")

py_model = "google/siglip-base-patch16-224"
js_fp32 = "Xenova/siglip-base-patch16-224"

# サンプル画像5枚のIDを取得（各カテゴリから1枚）
sample_ids = conn.execute("""
    SELECT DISTINCT ON (c.category) c.id, c.category
    FROM image_catalog c
    JOIN image_embeddings e ON c.id = e.id
    WHERE e.model_name = ?
    ORDER BY c.category, c.id
""", [py_model]).fetchdf()

print("クロス環境検索テスト（Top-5 一致度）")
print("=" * 70)
print("各環境内で同一画像のTop-5近傍を検索し、Python結果との一致度を確認\n")

js_models = [
    "Xenova/siglip-base-patch16-224",
    "Xenova/siglip-base-patch16-224-fp16",
    "Xenova/siglip-base-patch16-224-q8",
    "Xenova/siglip-base-patch16-224-q4",
]

for _, row in sample_ids.iterrows():
    query_id = row["id"]
    
    # Python モデルでの Top-5
    py_top5 = conn.execute("""
        SELECT e2.id,
               list_cosine_similarity(e1.embedding, e2.embedding) as sim
        FROM image_embeddings e1
        JOIN image_embeddings e2 ON e2.model_name = e1.model_name
        WHERE e1.id = ? AND e1.model_name = ?
          AND e2.id != e1.id
        ORDER BY sim DESC
        LIMIT 5
    """, [query_id, py_model]).fetchdf()
    py_top5_ids = set(py_top5["id"])
    
    print(f"Query: {query_id[:8]}... ({row['category']})")
    
    for js_model in js_models:
        js_top5 = conn.execute("""
            SELECT e2.id,
                   list_cosine_similarity(e1.embedding, e2.embedding) as sim
            FROM image_embeddings e1
            JOIN image_embeddings e2 ON e2.model_name = e1.model_name
            WHERE e1.id = ? AND e1.model_name = ?
              AND e2.id != e1.id
            ORDER BY sim DESC
            LIMIT 5
        """, [query_id, js_model]).fetchdf()
        js_top5_ids = set(js_top5["id"])
        
        overlap = len(py_top5_ids & js_top5_ids)
        short_name = js_model.split("/")[-1]
        print(f"  vs {short_name:>40s}: overlap={overlap}/5")
    print()

conn.close()

クロス環境検索テスト（Top-5 一致度）
各環境内で同一画像のTop-5近傍を検索し、Python結果との一致度を確認

Query: 01afe57e... (EuroPython2025)
  vs                  siglip-base-patch16-224: overlap=2/5
  vs             siglip-base-patch16-224-fp16: overlap=2/5
  vs               siglip-base-patch16-224-q8: overlap=3/5
  vs               siglip-base-patch16-224-q4: overlap=4/5

Query: 065d540f... (KashiwaVillagePark2026)
  vs                  siglip-base-patch16-224: overlap=4/5
  vs             siglip-base-patch16-224-fp16: overlap=4/5
  vs               siglip-base-patch16-224-q8: overlap=4/5
  vs               siglip-base-patch16-224-q4: overlap=4/5

Query: 00ffdb48... (PyConJP2025)
  vs                  siglip-base-patch16-224: overlap=4/5
  vs             siglip-base-patch16-224-fp16: overlap=4/5
  vs               siglip-base-patch16-224-q8: overlap=4/5
  vs               siglip-base-patch16-224-q4: overlap=3/5

Query: 027fe5cf... (PyConJP2025-PreCampHiroshima)
  vs                  siglip-base-patch16-224: overlap=2/5
  vs 

  vs               siglip-base-patch16-224-q8: overlap=3/5
  vs               siglip-base-patch16-224-q4: overlap=3/5



## 総合評価と考察

### DuckDB 保存結果

| model_name | dtype | 環境 | 画像数 |
|------------|-------|------|--------|
| `google/siglip-base-patch16-224` | fp32 | Python (PyTorch) | 378 |
| `openai/clip-vit-large-patch14` | fp32 | Python (PyTorch) | 378 |
| `Xenova/siglip-base-patch16-224` | fp32 | Node.js (ONNX) | 378 |
| `Xenova/siglip-base-patch16-224-fp16` | fp16 | Node.js (ONNX) | 378 |
| `Xenova/siglip-base-patch16-224-q8` | q8 | Node.js (ONNX) | 378 |
| `Xenova/siglip-base-patch16-224-q4` | q4 | Node.js (ONNX) | 378 |
| `Xenova/siglip-base-patch16-224-browser` | fp32 | Chromium (WASM) | 50 |

合計 **2,318 行**（7モデル）。既存の `image_embeddings` テーブルの複合PK `(id, model_name)` を活用し、同一テーブル内でモデル別にフィルタ可能。

### サニティチェック: JSON → DuckDB の完全一致を確認
全5モデル × 全画像で `max_diff = 0.00e+00` — DuckDB の FLOAT[768] 型への格納で精度劣化はない。

### クロス環境 Top-5 検索テスト結果

各カテゴリから1枚ずつ（計6枚）で Python と JS の Top-5 近傍の重なりを確認:

| カテゴリ | fp32 | fp16 | q8 | q4 |
|---------|------|------|----|----|
| EuroPython2025 | 2/5 | 2/5 | 3/5 | 4/5 |
| KashiwaVillagePark | 4/5 | 4/5 | 4/5 | 4/5 |
| PyConJP2025 | 4/5 | 4/5 | 4/5 | 3/5 |
| PreCampHiroshima | 2/5 | 2/5 | 1/5 | 1/5 |
| TokyoNight | 3/5 | 3/5 | 3/5 | 3/5 |
| terada | 4/5 | 4/5 | 3/5 | 3/5 |

- **平均 overlap: fp32/fp16 ≈ 3.2/5, q8 ≈ 3.0/5, q4 ≈ 3.0/5**
- Top-5 の重なりが 2-4/5 程度なのは、Notebook 33 の P@5 = 0.51 と整合する（Python と JS で半数程度の近傍が入れ替わる）
- ただし入れ替わる画像も多くは「僅差の近傍」であり、大きくランキングが崩れているわけではない

### Python / JS 間の差異の原因

Python vs JS fp32 の cosine similarity が 0.906 にとどまる原因を調査した結果、**画像リサイズ処理の実装差**が唯一にして最大の要因であることが判明した。

#### 前処理設定は同一
Python (transformers) と JS (Transformers.js) は同じ `preprocessor_config.json` を参照しており、設定値は完全に一致している:
- リサイズ: 224×224、bicubic 補間
- 正規化: `(pixel / 255 - 0.5) / 0.5` → 値域 [-1.0, 1.0]

#### リサイズの実装が異なる
- **Python**: PIL の `Image.resize(size, Image.Resampling.BICUBIC)` を使用
- **JS**: Transformers.js の内部コード（`src/utils/image.js`）で Sharp の `img.affine()` を使用

Transformers.js は bicubic リサイズに `sharp.affine()`（アフィン変換）を使っており、PIL の `Image.resize()` とは補間の内部実装が異なる。

#### 定量的な影響（500×666 → 224×224 の例）

| リサイズ方式 | ピクセル平均差 (uint8) | 最大差 | ずれるピクセルの割合 | 正規化後 cosine sim |
|------------|---------------------|--------|------------------|-------------------|
| `sharp.affine()` (現状の JS) | **4.44** | **124** / 255 | **88.6%** | 0.990 |
| `sharp.resize(kernel:'cubic')` | 0.19 | 3 / 255 | 19.0% | 0.99997 |

`sharp.affine()` では約9割のピクセルで値がずれ、最大で 124/255 もの差が出る。`sharp.resize()` を使えばほぼ PIL と一致する。

#### なぜリサイズ差が embedding に大きく影響するか
SigLIP (Vision Transformer) は 224×224 の入力画像を 16×16 のパッチに分割し、各パッチのピクセル値を直接 linear projection に通す。ピクセル値のずれは:
1. 196パッチのほぼ全てで入力が変わる
2. 12層の self-attention を通じて差が増幅される
3. 特にエッジやテクスチャなどの高周波成分で影響が大きい（夜景やカンファレンス写真で cosine sim が低くなる傾向と一致）

#### 対策
- **JS に統一して利用すれば前処理差の問題は発生しない**（JS 同士なら同じ前処理パイプラインを通る）
- Python との互換性が必要な場合は、Transformers.js のリサイズ処理を `sharp.resize()` に変更するパッチが有効

### DuckDB に保存したことで可能になること
1. **SQL でのクロスモデル比較**: `list_cosine_similarity()` を使って任意のモデル間で類似度を計算可能
2. **Streamlit アプリでの利用**: `model_name` パラメータで Python/JS/各量子化のベクトルを切り替えて検索
3. **将来の追加モデルとの比較**: 同じテーブル構造に新しいモデルのベクトルを追加するだけで比較可能
4. **HNSW インデックスの活用**: 既存のインデックスが全モデルに適用される（ただし `model_name` フィルタ付き検索ではフルスキャンになる点に注意）

### 「全て JS で運用する」場合のデータ管理
- **推奨**: `Xenova/siglip-base-patch16-224` (fp32) または `Xenova/siglip-base-patch16-224-fp16` をメインのインデックスとして使用
- Python の `google/siglip-base-patch16-224` は比較用の参照データとして保持
- 新しい画像の追加時は Node.js の `embed_images.mjs` で生成し、同じスキーマで DuckDB に挿入